In [23]:
%pip install didppy==0.11.1 cplex==22.2.0.1 docplex==2.32.264 six==1.17.0


[notice] A new release of pip is available: 25.2 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [24]:
import didppy
import cplex
import docplex
import six
import numpy as np
from docplex.cp.model import CpoModel

print("All packages installed successfully!")

All packages installed successfully!


replace this path with the location of your cpoptimizer

In [ ]:
context_solver_path = ""

In [30]:
# Paper toy instance (Fig. 1): m=3 machines, n=4 jobs, RCP optimum 553.
# This CP is a flow-shop with consecutive travel only (no single-robot contention),
# so it is a relaxation of the robotic cell; the makespan can be below 553.
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
sys.path.insert(0, str(ROOT))
from instance.instance_reader import read_instance

INSTANCE = ROOT / "instance" / "paper_sample_instance.txt"
m_paper, n_paper, p_pad, tau_st = read_instance(INSTANCE)

cp_model = CpoModel()

n = n_paper  # jobs
m = m_paper  # machines

J = list(range(1, n + 1))
M = list(range(1, m + 1))

# processing time of job j on machine i; rows = machines, columns = jobs
p = np.array([[p_pad[i][j] for j in J] for i in M], dtype=int)

# travel time from machine i to i+1 (paper τ, same for every job)
tau = np.array(
    [[tau_st[i][i + 1] for _ in J] for i in range(1, m)],
    dtype=int,
)

# greedy flow-shop schedule as a time-horizon UB (this CP has no single-robot constraint)
machine_available = [0] * m
for j in J:
    previous_completion = 0
    for i in range(m):
        arrival = 0 if i == 0 else previous_completion + int(tau[i - 1, j - 1])
        start_time = max(machine_available[i], arrival)
        finish_time = start_time + int(p[i, j - 1])
        machine_available[i] = finish_time
        previous_completion = finish_time
H = max(machine_available)
T = range(H)  # time periods: 0, 1, ..., H-1

# starting time of job j on machine i
s = {
    (i, j): cp_model.integer_var(
        min=0,
        max=H,
        name=f"s_{i}{j}"
    )
    for i in M
    for j in J
}

# completion time of job j on machine i
c = {
    (i, j): s[i, j] + int(p[i-1, j-1])
    for i in M
    for j in J
}


# binary decision variable x_ijt:
# 1 if job j starts processing on machine i at time t
# 0 otherwise
x = {
    (i, j, t): cp_model.binary_var(
        name=f"x_{i}{j}{t}"
    )
    for i in M
    for j in J
    for t in T
}

# constraint 2: very job starts exactly once on each machine
for i in M:
    for j in J:
        cp_model.add(
            cp_model.sum(
                x[i, j, t] for t in T
            ) == 1
        )

# constraint 2: each machine processes at most one job at a time
for i in M:
    for t in T:
        cp_model.add(
            cp_model.sum(
                x[i, j, t_prime]
                for j in J
                for t_prime in range(
                    max(0, t - int(p[i-1, j-1]) + 1),
                    t + 1
                )
            ) <= 1
        )

# link starting times to binary dec var
for i in M:
    for j in J:
        cp_model.add(
            s[i, j] == cp_model.sum(
                t * x[i, j, t] for t in T
            )
        )

# constraint _: ensure all jobs finish within the time horizon 
# not sure if we need this
for i in M:
    for j in J:
        cp_model.add(c[i, j] <= H)

# constraint 4: job j must finish on machine i and travel before starting on machine i+1
for i in M[:-1]:
    for j in J:
        cp_model.add(
            c[i, j] + int(tau[i-1, j-1])
            <= s[i+1, j]
        )

# # Constraint 5: Set x equal to 1 at time 0 -- constraint 5 doesn't seem right in that the jobs probably shouldn't ALL start at t = 0

# for i in M:
#     for j in J:
#         cp_model.add(x[i, j, 0] == 1)

# constraint 6: set processing times of dummy machines to zero
p_extended = np.zeros((m + 2, n), dtype=int)

# copy actual processing times into the middle rows
p_extended[1:m+1, :] = p

# minimize flow-shop makespan (last machine completion). RCP paper optimum is 553.
makespan = cp_model.integer_var(min=0, max=H, name="makespan")
for j in J:
    cp_model.add(c[m, j] <= makespan)
cp_model.add(cp_model.minimize(makespan))

print("Instance:", INSTANCE)
print("Number of jobs:", n)
print("Number of machines:", m)
print("Time horizon:", H)
print("Number of binary decision variables:", m * n * H)
print("Processing times:\n", p)
print("Travel times:\n", tau)


Number of jobs: 3
Number of machines: 2
Time horizon: 30
Number of binary decision variables: 180
Processing times:
 [[1 8 7]
 [5 5 9]]
Travel times:
 [[1 7 3]]


In [31]:
from docplex.cp.config import context

context.solver.agent = "local"
context.solver.local.execfile = context_solver_path

cp_model_solution = cp_model.solve(TimeLimit=60)

 ! --------------------------------------------------- CP Optimizer 22.2.0.0 --
 ! Satisfiability problem - 186 variables, 81 constraints
 ! Presolve      : 72 extractables eliminated
 ! Initial process time : 0.01s (0.01s extraction + 0.00s propagation)
 !  . Log search space  : 144.3 (before), 144.3 (after)
 !  . Memory usage      : 406.5 kB (before), 406.5 kB (after)
 ! Using parallel search with 10 workers.
 ! ----------------------------------------------------------------------------
 !               Branches  Non-fixed    W       Branch decision
                     1000          8    1   F     1  = x_231
 *                    389  0.02s        2         1  = x_2316
 ! ----------------------------------------------------------------------------
 ! Search completed, 1 solution found.
 ! ----------------------------------------------------------------------------
 ! Number of branches     : 4695
 ! Number of fails        : 2155
 ! Total memory usage     : 9.6 MB (9.5 MB CP Optimiz

In [32]:
print('Solution status: ', cp_model_solution.get_solve_status())
if cp_model_solution:
    print("Solution found!")
else:
    print("No solution found.")


Solution status:  Feasible
Solution found!


In [33]:
cp_model_x_soln = np.zeros((m, n, H), dtype=int)

for i in M:
    for j in J:
        for t in T:
            cp_model_x_soln[i-1, j-1, t] = int(
                round(cp_model_solution.get_value(x[i, j, t]))
            )

print("=" * 50)
print("DECISION VARIABLES (x)")
print("=" * 50)

for i in M:
    for j in J:
        start_periods = [
            t for t in T if cp_model_x_soln[i-1, j-1, t] == 1
        ]
        print(f"Machine {i}, Job {j}: x = 1 at time(s) {start_periods}")

DECISION VARIABLES (x)
Machine 1, Job 1: x = 1 at time(s) [1]
Machine 1, Job 2: x = 1 at time(s) [10]
Machine 1, Job 3: x = 1 at time(s) [3]
Machine 2, Job 1: x = 1 at time(s) [3]
Machine 2, Job 2: x = 1 at time(s) [25]
Machine 2, Job 3: x = 1 at time(s) [16]


In [34]:
cp_model_s_soln = np.zeros((m, n), dtype=int)

for i in M:
    for j in J:
        cp_model_s_soln[i-1, j-1] = cp_model_solution.get_value(s[i, j])

print("\n" + "=" * 50)
print("START TIMES (s)")
print("=" * 50)

for i in M:
    for j in J:
        print(f"Job {j} starts on Machine {i} at time {cp_model_s_soln[i-1, j-1]}")


START TIMES (s)
Job 1 starts on Machine 1 at time 1
Job 2 starts on Machine 1 at time 10
Job 3 starts on Machine 1 at time 3
Job 1 starts on Machine 2 at time 3
Job 2 starts on Machine 2 at time 25
Job 3 starts on Machine 2 at time 16


In [35]:
cp_model_c_soln = np.zeros((m, n), dtype=int)

for i in M:
    for j in J:
        start_time = cp_model_s_soln[i-1, j-1]
        processing_time = p[i-1, j-1]

        cp_model_c_soln[i-1, j-1] = start_time + processing_time

print("\n" + "=" * 50)
print("COMPLETION TIMES (c)")
print("=" * 50)

for i in M:
    for j in J:
        print(f"Job {j} finishes on Machine {i} at time {cp_model_c_soln[i-1, j-1]}")


COMPLETION TIMES (c)
Job 1 finishes on Machine 1 at time 2
Job 2 finishes on Machine 1 at time 18
Job 3 finishes on Machine 1 at time 10
Job 1 finishes on Machine 2 at time 8
Job 2 finishes on Machine 2 at time 30
Job 3 finishes on Machine 2 at time 25


In [36]:
print("\n" + "=" * 50)
print("COMPLETE SCHEDULE")
print("=" * 50)

for i in M:
    print(f"\nMachine {i}:")

    # Sort jobs by their start times
    jobs_sorted = sorted(
        J,
        key=lambda j: cp_model_s_soln[i-1, j-1]
    )

    for j in jobs_sorted:
        start = cp_model_s_soln[i-1, j-1]
        finish = cp_model_c_soln[i-1, j-1]
        duration = p[i-1, j-1]

        print(
            f"  Job {j}: Start = {start}, "
            f"Finish = {finish}, "
            f"Processing Time = {duration}"
        )


COMPLETE SCHEDULE

Machine 1:
  Job 1: Start = 1, Finish = 2, Processing Time = 1
  Job 3: Start = 3, Finish = 10, Processing Time = 7
  Job 2: Start = 10, Finish = 18, Processing Time = 8

Machine 2:
  Job 1: Start = 3, Finish = 8, Processing Time = 5
  Job 3: Start = 16, Finish = 25, Processing Time = 9
  Job 2: Start = 25, Finish = 30, Processing Time = 5


In [37]:
print("\n" + "=" * 50)
print("OBJECTIVE VALUE")
print("=" * 50)

print("Objective value:", cp_model_solution.get_objective_value())


OBJECTIVE VALUE
Objective value: None
